In [2]:
import os
import requests
import logging
import time
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
import csv
import pandas as pd 
import numpy as np 
from tqdm.notebook import tqdm
from eutils import EutilsNCBIError, EutilsRequestError
from metapub import PubMedFetcher, pubmedcentral
from datetime import datetime
from tqdm.auto import tqdm
#Import DR module from Functions folder
from Functions import DataRetrieval as DR
#API_KEY
from Reference_files.keys import API_KEY as api_key
import urllib
import json

In [3]:
# Initialize logger
prefix = "test"+str(datetime.now()).split()[0]
file_handler = logging.FileHandler(f"{prefix}_Examples.log", mode='w')
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
logging.getLogger().addHandler(file_handler)

# From Query to pubmedCentral full text

In [4]:
# Initialize PubMed fetcher
fetcher = PubMedFetcher()

# Query construction from filters

In [5]:
O = '''("english"[Language] 
NOT "meta-analysis"[Publication Type] 
NOT "review"[Publication Type] 
NOT "retracted publication"[Publication Type] 
NOT "retraction of publication"[Publication Type] 
NOT "published erratum"[Publication Type] 
NOT "controlled clinical trial"[Publication Type] 
NOT "clinical study"[Publication Type] 
NOT "clinical trial"[Publication Type] 
NOT "clinical trial protocol"[Publication Type] 
NOT "clinical trial, phase i"[Publication Type] 
NOT "clinical trial, phase ii"[Publication Type] 
NOT "clinical trial, phase iii"[Publication Type] 
NOT "clinical trial, phase iv"[Publication Type] 
NOT "clinical trial, veterinary"[Publication Type])'''

A1 = '''"nucleoproteins"[MeSH Terms] 
OR "protein interaction mapping"[MeSH Terms] OR ("nucleoprotein"[All Fields] 
OR "nucleoproteins"[All Fields] 
OR "multiprotein"[All Fields] 
OR "multiproteins"[All Fields] 
OR "proteins"[MeSH Terms] 
OR "protein"[All Fields] 
OR "proteins"[All Fields] 
OR "enzyme"[All Fields]) 
AND 
("interact"[All Fields] 
OR "interacted"[All Fields] 
OR "interacting"[All Fields] 
OR "interaction"[All Fields] 
OR "interactions"[All Fields] 
OR "interactivity"[All Fields] 
OR "interacts"[All Fields]) OR ("protein interaction"[All Fields] 
OR "protein interactions"[All Fields] 
OR "interacting protein"[All Fields] 
OR "interacting proteins"[All Fields] OR "multiprotein complexes"[MeSH Terms]) OR
("nucleoprotein"[All Fields] 
OR "nucleoproteins"[All Fields] 
OR "multiprotein"[All Fields] 
OR "multiproteins"[All Fields] 
OR "proteins"[MeSH Terms] 
OR "protein"[All Fields] 
OR "proteins"[All Fields] 
OR "enzyme"[All Fields]) 
AND 
("complex"[All Fields] 
OR "complexes"[All Fields] 
OR "heteromer"[All Fields] 
OR "heteromers"[All Fields] 
OR "homomer"[All Fields] 
OR "homomers"[All Fields] 
OR "heteromeric"[All Fields] 
OR "homomeric"[All Fields] 
OR "subunit"[All Fields] 
OR "subunits"[All Fields])
OR "protein complex"[All Fields] 
OR "protein complexes"[All Fields] OR ("protein"[All Fields] 
AND ("RNA"[All Fields] 
OR "DNA"[All Fields] 
OR "ribonucleic"[All Fields] 
OR "deoxyribonucleic"[All Fields]) 
AND ("interaction"[All Fields] 
OR "interactions"[All Fields] ))'''

A2 = '''"Immunoprecipitation"[Mesh] 
OR coimmunoprecipitation 
OR ("co"[All Fields] AND "immunoprecipitation"[All Fields]) 
OR ("RNA" [All Fields] AND "immunoprecipitation" [All Fields]) 
OR "co immunoprecipitation"[All Fields] OR "coIP"[All Fields] OR "Chromatography, Affinity"[Mesh] OR "affinity purification"[All Fields] OR "affinity isolation"[All Fields] OR "affinity chromatography"[All Fields] 
OR ("affinity"[All Fields] AND ("purification"[All Fields] OR "isolation"[All Fields] OR "chromatography"[All Fields])) 
OR("pull"[All Fields] AND ("down"[All Fields] OR "downs"[All Fields]))'''


A3 = '''"crystallography, x ray"[MeSH Terms]
OR "Nuclear Magnetic Resonance, Biomolecular"[MeSH Terms]
OR "Cryoelectron Microscopy"[MeSH Terms]
OR "Protein Array Analysis"[MeSH Terms]
OR "electrophoretic mobility shift assay"[MeSH Terms] 
OR "Surface Plasmon Resonance"[MeSH Terms]'''

B = '''"Epitope Mapping"[MeSH Terms] 
OR "Precipitin Tests"[MeSH Terms] 
OR "Two-Hybrid System Techniques"[MeSH Terms] 
OR "blotting, far western"[MeSH Terms] 
OR "Radioimmunoprecipitation Assay"[MeSH Terms] 
OR "Autoantibodies"[MeSH Terms] 
OR "Chromatin Immunoprecipitation"[MeSH Terms] 
OR "Cross-Linking Reagents"[MeSH Terms] 
OR "Formaldehyde"[MeSH Terms] 
OR "Microscopy"[MeSH Terms]OR (crosslink AND reagent) 
OR "Formaldehyde"[All Fields] OR "DNA Footprinting"[Mesh] 
OR "Nuclease Protection Assays"[Mesh]  
OR "Blotting, Southwestern"[Mesh] 
OR "Fluorescence Resonance Energy Transfer"[Mesh] 
OR "PAR CLIP"[All Fields]
OR "AlphaFold"[All Fields]'''



query = f"({O} AND ({A1} OR {A3})) NOT ({A1} AND {B} NOT ({A2} AND {A3}))"

In [6]:
query_file = "./Reference_files/query.txt" # in case query is in a file
query = DR.read_query_from_file(query_file)

In [7]:
start_date = "2000-01-01"
stop_date = None  # Will default to the current date if None\
pmid_list, intervals = DR.fetch_pmids_over_period(query, start=start_date, stop=stop_date, full_text=True, plot=True, max_workers=1)

Generating date batches from 2000-01-01 to 2025-07-01...

2000-01-01 to 2000-12-31 → 1071 papers
2001-01-01 to 2002-01-01 → 2466 papers
2002-01-02 to 2003-01-02 → 2803 papers
2003-01-03 to 2004-01-03 → 3398 papers
2004-01-04 to 2005-01-03 → 4184 papers
2005-01-04 to 2006-01-04 → 4986 papers
2006-01-05 to 2007-01-05 → 6089 papers
2007-01-06 to 2008-01-06 → 9489 papers
2008-01-07 to 2009-01-06 → 13691 papers
2008-01-07 to 2008-07-05 → 6031 papers
2008-07-06 to 2009-07-06 → 13691 papers
2008-07-06 to 2009-01-02 → 13691 papers
2008-07-06 to 2008-10-04 → 6031 papers
2008-10-05 to 2009-10-05 → 13691 papers
2008-10-05 to 2009-04-03 → 13691 papers
2008-10-05 to 2009-01-03 → 13691 papers
2008-10-05 to 2008-12-04 → 6031 papers
2008-12-05 to 2009-12-05 → 13691 papers
2008-12-05 to 2009-06-03 → 13691 papers
2008-12-05 to 2009-03-05 → 13691 papers
2008-12-05 to 2009-02-03 → 13691 papers
2008-12-05 to 2009-01-14 → 13691 papers
2008-12-05 to 2008-12-25 → 6031 papers
2008-12-26 to 2009-12-26 → 13691 p

2025-07-01 13:31:25 littlebeauty root[3198497] WARNING No acceptable window found at 2009-12-31, forcing fallback window.


2009-12-31 to 2010-12-31 → 17404 papers
2009-12-31 to 2010-06-29 → 17404 papers
2009-12-31 to 2010-03-31 → 17404 papers
2009-12-31 to 2010-03-01 → 17404 papers
2009-12-31 to 2010-02-09 → 17404 papers
2009-12-31 to 2010-01-20 → 17404 papers
2009-12-31 to 2010-01-10 → 17404 papers
2009-12-31 to 2010-01-01 → 17404 papers
2010-01-02 to 2011-01-02 → 21107 papers
2010-01-02 to 2010-07-01 → 9744 papers
2010-07-02 to 2011-07-02 → 21107 papers
2010-07-02 to 2010-12-29 → 9744 papers
2010-12-30 to 2011-12-30 → 21107 papers
2010-12-30 to 2011-06-28 → 21107 papers
2010-12-30 to 2011-03-30 → 21107 papers
2010-12-30 to 2011-02-28 → 21107 papers
2010-12-30 to 2011-02-08 → 21107 papers
2010-12-30 to 2011-01-19 → 21107 papers
2010-12-30 to 2011-01-09 → 21107 papers
2010-12-30 to 2010-12-31 → 9744 papers


2025-07-01 13:32:03 littlebeauty root[3198497] WARNING No acceptable window found at 2011-01-01, forcing fallback window.


2011-01-01 to 2012-01-01 → 25750 papers
2011-01-01 to 2011-06-30 → 11363 papers
2011-01-01 to 2011-04-01 → 11363 papers
2011-01-01 to 2011-03-02 → 11363 papers
2011-01-01 to 2011-02-10 → 11363 papers
2011-01-01 to 2011-01-21 → 11363 papers
2011-01-01 to 2011-01-11 → 11363 papers
2011-01-01 to 2011-01-02 → 11363 papers


...

2025-07-01 13:33:06 littlebeauty root[3198497] WARNING No acceptable window found at 2011-01-13, forcing fallback window.


2011-01-13 to 2012-01-13 → 25750 papers
2011-01-13 to 2011-07-12 → 11363 papers
2011-01-13 to 2011-04-13 → 11363 papers
2011-01-13 to 2011-03-14 → 11363 papers
2011-01-13 to 2011-02-22 → 11363 papers
2011-01-13 to 2011-02-02 → 11363 papers
2011-01-13 to 2011-01-23 → 11363 papers
2011-01-13 to 2011-01-14 → 11363 papers


2025-07-01 13:33:17 littlebeauty root[3198497] WARNING No acceptable window found at 2011-01-15, forcing fallback window.


2011-01-15 to 2012-01-15 → 25750 papers
2011-01-15 to 2011-07-14 → 11363 papers
2011-01-15 to 2011-04-15 → 11363 papers
2011-01-15 to 2011-03-16 → 11363 papers
2011-01-15 to 2011-02-24 → 11363 papers
2011-01-15 to 2011-02-04 → 11363 papers
2011-01-15 to 2011-01-25 → 11363 papers
2011-01-15 to 2011-01-16 → 11363 papers


KeyboardInterrupt: 

In [9]:
pd.DataFrame(pmid_list).to_csv("pmid_list.txt", index=False)

In [ ]:
def plot_from_metadata(metadata):
    """Visualize counts using metadata from fetch_pmids_parallel."""
    df = pd.DataFrame(metadata)
    df['Date Range'] = df['start'] + '\nto\n' + df['end']
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(df['Date Range'], df['count'], color='skyblue')
    
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom')
    
    plt.title("Paper Counts by Date Range")
    plt.xlabel("Date Range")
    plt.ylabel("Number of Papers")
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    
    plt.show()


In [ ]:

def plot_density_over_time(batch_metadata):
    if not batch_metadata:
        print("No batch metadata to plot.")
        return

    mid_dates = []
    densities = []

    for meta in batch_metadata:
        start = datetime.fromisoformat(meta["start"])
        end = datetime.fromisoformat(meta["end"])
        duration = (end - start).days or 1
        mid = start + (end - start) / 2
        mid_dates.append(mid)
        densities.append(meta["count"] / duration)

    plt.figure(figsize=(12, 6))
    plt.plot(mid_dates, densities, marker='o', linestyle='-', color='blue')
    plt.title("Paper count over time")
    plt.xlabel("Date")
    plt.ylabel("Papers per Day")
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [8]:
def plot_intervals(intervals, count_col='count', start_col='start', end_col='end', 
                   title='Count Over Date Ranges (PA Plot Style)', 
                   xlabel='Date', ylabel='Count', color='blue', linewidth=4,
                   figsize=(12, 6)):
    """
    Plot intervals in PA Plot style (horizontal lines representing date ranges with counts).
    
    Parameters:
    - intervals: DataFrame containing the interval data
    - count_col: Column name for count values (default 'count')
    - start_col: Column name for start dates (default 'start')
    - end_col: Column name for end dates (default 'end')
    - title: Plot title (default 'Count Over Date Ranges (PA Plot Style)')
    - xlabel: X-axis label (default 'Date')
    - ylabel: Y-axis label (default 'Count')
    - color: Line color (default 'blue')
    - linewidth: Line width (default 4)
    - figsize: Figure size (default (12, 6))
    """
    # Convert to datetime if not already
    intervals[start_col] = pd.to_datetime(intervals[start_col])
    intervals[end_col] = pd.to_datetime(intervals[end_col])
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot each interval
    for idx, row in intervals.iterrows():
        ax.plot([row[start_col], row[end_col]], [row[count_col]] * 2, 
                color=color, linewidth=linewidth)
    
    # Formatting
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    
    return fig, ax

In [12]:
type(intervals)

list

In [11]:
plot_intervals(intervals, color='red', linewidth=2, title='Publication Counts', ylabel='Paper Count')
plt.show()

TypeError: list indices must be integers or slices, not str

In [ ]:
pmc_id_list = DR.get_pmcid_for_otherid(pmid_list)

In [ ]:
oa_pmcids = DR.filter_oa_database(pmc_id_list) # downloads open access database file from NCBI server \size: ~230 mgb\

In [ ]:
## Fetch openaccess PMCs from a query
start_date = "2000-01-01"
stop_date = None  # Will default to the current date if None
pmid_array = DR.fetch_pmids_over_period(query, start=start_date, stop_date=stop_date, full_text = True)
pmc_id_list = DR.get_pmcid_for_otherid(pmid_list)
oa_pmcids = DR.filter_oa_database(pmc_id_list) # downloads open access database file from NCBI server \size: ~230 mgb\

## Download papers using python requests

In [6]:
oa_pmcids = ["PMC9770967"]

In [7]:
success_count, failed = DR.download_pmc_articles(oa_pmcids)


Download Summary:
Successfully processed 1/1 full-text files


###  Or download using powershell \windows/faster\ 

In [ ]:
# Save the oa_pmcids pandas series to a text file that powershell can read(one ID per line)
mkdir -p ./Full_text_jsons

# Read PMCIDs from the file and process each one
Get-Content oa_pmcids.txt | ForEach-Object {
    $pmcid = $_

    # 1. Download full-text JSON
    $jsonUrl = "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/$pmcid/unicode"
    $jsonOut = "./Full_text_jsons/$pmcid.json"
    Invoke-RestMethod -Uri $jsonUrl -OutFile $jsonOut

    # 2. Download supplementary ZIP
    $zipUrl = "https://www.ebi.ac.uk/europepmc/webservices/rest/$pmcid/supplementaryFiles"
    $zipOut = "./Full_text_jsons/${pmcid}_supp.zip"
    Invoke-WebRequest -Uri $zipUrl -OutFile $zipOut
}

# Or Using bash

In [ ]:
%%bash

mkdir -p ./Full_text_jsons

while IFS= read -r PMCID; do
    echo "Processing $PMCID"

    # 1. Download full-text JSON
    json_url="https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/${PMCID}/unicode"
    curl -s "${json_url}" -o "./Full_text_jsons/${PMCID}.json"

    # 2. Download supplementary ZIP
    zip_url="https://www.ebi.ac.uk/europepmc/webservices/rest/${PMCID}/supplementaryFiles"
    curl -s "${zip_url}" -o "./Full_text_jsons/${PMCID}_supp.zip"
done < full_text_pmc.txt


# Paper Metadata

In [ ]:
paper_meta = DR.fetch_articles_meta(pmid_list)

### add publisher if needed using Xreff

In [ ]:
added_publishers = DR.process_publishers(paper_meta, email="your_email@rcf.edu")

In [ ]:
import random 
pmid_list = pmid_list.tolist()
random_ppers = random.sample(pmid_list, 50)

In [ ]:
paper_meta_50 = DR.fetch_articles_meta(random_ppers)

# process full text json and parse relevant data into a dataframe 

In [8]:
section_types = ['TITLE', 'ABSTRACT', 'INTRO', 'METHODS', 'RESULTS', 'CONCL', 'FIG', 'SUPPL', 'DISCUSS']
full_text_df = DR.extract_text_from_json_to_dataframe("./Full_text_jsons", section_types)

2025-05-28 18:50:12 LAPTOP-S8N3C7A8 Functions.DataRetrieval[21192] INFO Extracting from JSON files in ./Full_text_jsons
2025-05-28 18:50:12 LAPTOP-S8N3C7A8 Functions.DataRetrieval[21192] INFO Processed 1 files, 0 errors
2025-05-28 18:50:12 LAPTOP-S8N3C7A8 Functions.DataRetrieval[21192] INFO Extracted 100 entries


### fetch aditional metadata that is not avalabble in the fulltext json eg. Journal, ISSN

In [9]:
meta_text = DR.add_metadata_to_dataframe(full_text_df)

Fetching metadata:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching metadata: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]


# BioC supplementary

# Creat full_text table + supplementary 

In [ ]:
full_text_df = DR.extract_text_from_json_to_dataframe("./XML+JSON", section_types, "./XML+JSON")